# Ablation Study — MBInception (ResNet-Inception) on CIFAR-100

Same architecture, same training pipeline as the CIFAR-10 version, and the same original 4 structural ablations plus a 7-way convolution-type sweep (11 configs total) —
only the dataset, normalization stats, and `num_classes` change. Keeping everything else
identical (base_filters, num_blocks, optimizer, schedule) means any gaps you see between the
CIFAR-10 and CIFAR-100 ablation results are attributable to the dataset, not to a mismatched setup.

Studies the contribution of each architectural component:
- Inception-style multi-branch blocks (`use_inception`)
- Convolution operator variants inside the 3x3/5x5 branches (`conv_type`: regular / separable / grouped / mixconv / asymmetric / ghost / bsconv)
- Residual / skip connections (`use_skip`)
- Stochastic depth / DropPath (`drop_path_rate`)

Run the **Quick test** cell first to confirm everything works before committing to the full run.

**Workflow:** instead of one long loop that trains all 6 configs back to back, each ablation has its own cell (Section 11) that trains just that one config and saves its result to disk immediately. Run the setup cells once (Sections 1–8), then run the config cells one at a time — in any order, in separate sessions, restarting the kernel between them if you need to. A final aggregation cell (Section 12) picks up whatever's been saved so far and builds the comparison plots/table once all 6 are done.

**Note on comparing to the CIFAR-10 run:** CIFAR-100 has the same 50,000/10,000 train/test image counts as CIFAR-10, but spreads them across 100 classes instead of 10 — 500 images/class instead of 5,000. So this isn't a "more data" experiment; it's a "harder, more fine-grained task on the same amount of data" experiment. Expect overall accuracy to be much lower than the CIFAR-10 run, and watch whether components that underperformed on CIFAR-10 (Inception, separable convs) do better here, since they were originally designed for exactly this kind of harder, larger-class-count setting.


## 1. Imports

In [ ]:
import csv
import json
import math
import os
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as tt

import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Guard against a GPU whose compute capability this PyTorch build doesn't
# actually support (e.g. Kaggle's P100 / sm_60 with a PyTorch build that only
# ships kernels for sm_70+). Without this check, every forward pass would
# crash with "CUDA error: no kernel image is available for execution on the
# device" instead of a clear message up front.
if device.type == "cuda":
    cap_major, cap_minor = torch.cuda.get_device_capability(0)
    cap_str = f"sm_{cap_major}{cap_minor}"
    supported = torch.cuda.get_arch_list()
    if cap_str not in supported:
        print(
            f"Warning: {torch.cuda.get_device_name(0)} has compute capability "
            f"{cap_str}, which this PyTorch build does not support "
            f"(supports: {supported}). Falling back to CPU.\n"
            f"On Kaggle: Settings -> Accelerator -> switch to a T4 GPU, then "
            f"restart the session and rerun."
        )
        device = torch.device("cpu")

print(f"Device: {device}")


## 2. Base model

Unchanged from the main notebook.

In [ ]:
class BaseModel(nn.Module):
    def __init__(self):
        super().__init__()

    def training_step(self, batch, l1_lambda: float = 0.0):
        images, targets = batch
        outputs = self(images)
        loss = F.cross_entropy(outputs, targets)
        if l1_lambda > 0:
            l1_norm = sum(p.abs().sum() for p in self.parameters())
            loss = loss + l1_lambda * l1_norm
        return loss

    def validation_step(self, batch):
        images, targets = batch
        outputs = self(images)
        loss = F.cross_entropy(outputs, targets)
        acc = (outputs.argmax(dim=1) == targets).float().mean()
        return {"val_loss": loss.detach(), "val_acc": acc}

    def validation_epoch_end(self, outputs):
        batch_losses = [x["val_loss"] for x in outputs]
        batch_accs = [x["val_acc"] for x in outputs]
        return {
            "val_loss": torch.stack(batch_losses).mean().item(),
            "val_acc": torch.stack(batch_accs).mean().item(),
        }

## 3. Building blocks

Each block that's under study takes an ablation toggle (`conv_type`, `use_skip`, etc.). `conv_type` swaps out the convolution used inside the Inception block's 3x3/5x5 branches (and inside the plain-conv fallback block) for one of several conv operators from the efficiency/architecture literature -- see the note right before `InceptionBlockLight` below.

In [ ]:
class DropPath(nn.Module):
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1.0 - self.drop_prob
        mask = torch.rand(x.shape[0], 1, 1, 1, device=x.device) < keep_prob
        return x.div(keep_prob) * mask


class SeparableConv2d(nn.Module):
    """Depthwise + pointwise convolution, with BN+ReLU between the two steps
    so the pair isn't just two back-to-back linear ops with no nonlinearity
    in between. (NOTE: this BN+ReLU is internal to the depthwise->pointwise
    transition only -- it is NOT the block-level BN/ReLU. ResNetBlock still
    owns all block-level BN/activation/residual handling; see the note in
    InceptionBlockLight/PlainConvBlock.)"""

    def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding=0):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_ch, in_ch, kernel_size, stride, padding, groups=in_ch, bias=False
        )
        self.bn = nn.BatchNorm2d(in_ch)
        self.act = nn.ReLU(inplace=True)
        self.pointwise = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.bn(x)
        x = self.act(x)
        x = self.pointwise(x)
        return x


class RegularConv2d(nn.Module):
    """Standard (non-separable) convolution, same interface as SeparableConv2d.
    Used for the 'no separable conv' ablation."""

    def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding=0):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding, bias=False)

    def forward(self, x):
        return self.conv(x)


### Convolution variants from the literature

`SeparableConv2d` (depthwise + pointwise) was the only alternative to a plain convolution in the
original ablation. The `conv_type` flag below extends this to a small menu of conv operators that
each attack the "how do we make a kxk convolution cheaper / more expressive" problem a different
way. All of them share the exact same constructor signature
(`in_ch, out_ch, kernel_size, stride, padding`) as `SeparableConv2d` / `RegularConv2d`, so they can
be dropped into `InceptionBlockLight`'s 3x3/5x5 branches and `PlainConvBlock` without touching
anything else in the model.

| `conv_type`  | Idea | Source |
|---|---|---|
| `regular`    | Plain dense kxk convolution (baseline) | -- |
| `separable`  | Depthwise kxk, then pointwise 1x1 | Howard et al. 2017, *MobileNets* |
| `grouped`    | kxk convolution split into `g` independent groups (cardinality) | Xie et al. 2017, *ResNeXt* |
| `mixconv`    | Channels split into groups, each with a **different** depthwise kernel size, concatenated and mixed with a pointwise conv | Tan & Le 2019, *MixConv* |
| `asymmetric` | kxk factorized into 1xk then kx1 | Szegedy et al. 2016, *Inception-v3* |
| `ghost`      | A few "intrinsic" feature maps from a small conv, the rest generated cheaply from those via depthwise convs | Han et al. 2020, *GhostNet* |
| `bsconv`     | Pointwise projection **first**, depthwise spatial mixing second (reverse of MobileNet's order) | Haase & Amthor 2020, *Blueprint Separable Convolutions* |


In [ ]:
def _pick_groups(in_ch, out_ch, max_groups=8):
    """Largest cardinality <= max_groups that evenly divides both channel
    counts, so grouped convolutions stay well-defined even for the small /
    unevenly-split channel counts the Inception branches can produce.
    Falls back down to 1 (a regular convolution) if nothing else divides."""
    g = math.gcd(in_ch, out_ch)
    for candidate in range(min(max_groups, g), 0, -1):
        if g % candidate == 0:
            return candidate
    return 1


def _split_channels(channels, num_groups):
    """Split `channels` into `num_groups` near-equal integer chunks that sum
    back to `channels` exactly (any remainder goes to the first chunk)."""
    split = [channels // num_groups] * num_groups
    split[0] += channels - sum(split)
    return split


class GroupedConv2d(nn.Module):
    """Grouped convolution a la ResNeXt (Xie et al., CVPR 2017, "Aggregated
    Residual Transformations for Deep Neural Networks"). Splits the
    in/out channels into `groups` independent paths -- a middle ground
    between a full dense convolution (groups=1) and a fully depthwise one
    (groups=in_ch) -- convolves each path separately, then concatenates.
    Cardinality (`groups`) is picked automatically per call via
    `_pick_groups` so it stays valid for whatever channel counts the
    surrounding block passes in."""

    def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding=0, groups=None):
        super().__init__()
        if groups is None:
            groups = _pick_groups(in_ch, out_ch)
        self.conv = nn.Conv2d(
            in_ch, out_ch, kernel_size, stride, padding, groups=groups, bias=False
        )

    def forward(self, x):
        return self.conv(x)


class MixConv2d(nn.Module):
    """Mixed depthwise convolution (Tan & Le, BMVC 2019, "MixConv: Mixed
    Depthwise Convolutional Kernels"). Splits the input channels into
    groups and gives each group a different depthwise kernel size (e.g.
    3x3 / 5x5 / 7x7 running in parallel), instead of committing every
    channel to one fixed receptive field. The concatenated groups are then
    mixed across channels with a pointwise convolution, mirroring the
    depthwise->pointwise structure of `SeparableConv2d`."""

    def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding=0, num_groups=3):
        super().__init__()
        num_groups = max(1, min(num_groups, in_ch))
        kernel_sizes = [kernel_size + 2 * g for g in range(num_groups)]
        self.splits = _split_channels(in_ch, num_groups)

        self.branches = nn.ModuleList()
        for k, ch in zip(kernel_sizes, self.splits):
            pad = k // 2
            self.branches.append(
                nn.Conv2d(ch, ch, k, stride, pad, groups=ch, bias=False)
            )

        self.bn = nn.BatchNorm2d(in_ch)
        self.act = nn.ReLU(inplace=True)
        self.pointwise = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)

    def forward(self, x):
        chunks = torch.split(x, self.splits, dim=1)
        out = torch.cat(
            [branch(chunk) for branch, chunk in zip(self.branches, chunks)], dim=1
        )
        out = self.act(self.bn(out))
        return self.pointwise(out)


class AsymmetricConv2d(nn.Module):
    """Spatially factorized ("asymmetric") convolution: a kxk kernel is
    replaced by a 1xk convolution followed by a kx1 convolution (Szegedy
    et al., CVPR 2016, "Rethinking the Inception Architecture for Computer
    Vision" -- Inception-v3). Cuts the per-channel-pair cost from k^2 to
    2k while keeping the same receptive field, trading off the assumption
    that the kxk filter is well approximated by a rank-1 spatial split.
    The stride is applied once, split across the two convs (width on the
    row conv, height on the column conv) so the net downsampling matches
    the other branches in the block exactly."""

    def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding=0):
        super().__init__()
        self.row_conv = nn.Conv2d(
            in_ch, out_ch, (1, kernel_size), stride=(1, stride),
            padding=(0, padding), bias=False,
        )
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)
        self.col_conv = nn.Conv2d(
            out_ch, out_ch, (kernel_size, 1), stride=(stride, 1),
            padding=(padding, 0), bias=False,
        )

    def forward(self, x):
        x = self.row_conv(x)
        x = self.bn(x)
        x = self.act(x)
        x = self.col_conv(x)
        return x


class GhostConv2d(nn.Module):
    """Ghost convolution (Han et al., CVPR 2020, "GhostNet: More Features
    from Cheap Operations"). A small regular convolution first produces a
    handful of "intrinsic" feature maps; cheap depthwise 3x3 convolutions
    then generate the rest ("ghost" feature maps) from those intrinsic
    ones, and the two sets are concatenated. Motivated by the observation
    that many feature maps in a trained CNN are near-redundant, near-linear
    transforms of one another, so most of them don't need a full
    convolution to produce. `cheap_ch` is always constructed as an exact
    multiple of `init_ch` so the grouped "cheap" conv is always valid,
    then the concatenated output is sliced down to `out_ch`."""

    def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding=0, ratio=2):
        super().__init__()
        init_ch = math.ceil(out_ch / ratio)
        cheap_ch = init_ch * (ratio - 1)
        self.out_ch = out_ch

        self.primary = nn.Sequential(
            nn.Conv2d(in_ch, init_ch, kernel_size, stride, padding, bias=False),
            nn.BatchNorm2d(init_ch),
            nn.ReLU(inplace=True),
        )
        self.cheap = nn.Sequential(
            nn.Conv2d(init_ch, cheap_ch, 3, 1, 1, groups=init_ch, bias=False),
            nn.BatchNorm2d(cheap_ch),
            nn.ReLU(inplace=True),
        ) if cheap_ch > 0 else None

    def forward(self, x):
        y = self.primary(x)
        if self.cheap is None:
            return y[:, : self.out_ch]
        ghost = self.cheap(y)
        return torch.cat([y, ghost], dim=1)[:, : self.out_ch]


class BSConv2d(nn.Module):
    """Blueprint Separable Convolution (Haase & Amthor, CVPR 2020,
    "Rethinking Depthwise Separable Convolutions: How Intra-Kernel
    Correlations Lead to Improved MobileNets"). Reverses MobileNet's
    order: a pointwise convolution mixes channels *first*, then a
    depthwise convolution handles spatial mixing. Motivated by the
    empirical finding that kernels in a trained CNN show strong
    correlation across their input-channel "planes", so a single
    pointwise combination captures most of what's needed and the
    depthwise step only has to learn per-channel spatial structure."""

    def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding=0):
        super().__init__()
        self.pointwise = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)
        self.depthwise = nn.Conv2d(
            out_ch, out_ch, kernel_size, stride, padding, groups=out_ch, bias=False
        )

    def forward(self, x):
        x = self.pointwise(x)
        x = self.bn(x)
        x = self.act(x)
        x = self.depthwise(x)
        return x


CONV_TYPES = ("regular", "separable", "grouped", "mixconv", "asymmetric", "ghost", "bsconv")


def make_conv(in_ch, out_ch, kernel_size, stride, padding, conv_type="separable"):
    if conv_type == "regular":
        return RegularConv2d(in_ch, out_ch, kernel_size, stride, padding)
    elif conv_type == "separable":
        return SeparableConv2d(in_ch, out_ch, kernel_size, stride, padding)
    elif conv_type == "grouped":
        return GroupedConv2d(in_ch, out_ch, kernel_size, stride, padding)
    elif conv_type == "mixconv":
        return MixConv2d(in_ch, out_ch, kernel_size, stride, padding)
    elif conv_type == "asymmetric":
        return AsymmetricConv2d(in_ch, out_ch, kernel_size, stride, padding)
    elif conv_type == "ghost":
        return GhostConv2d(in_ch, out_ch, kernel_size, stride, padding)
    elif conv_type == "bsconv":
        return BSConv2d(in_ch, out_ch, kernel_size, stride, padding)
    else:
        raise ValueError(f"Unknown conv_type '{conv_type}'. Options: {CONV_TYPES}")


In [ ]:
class InceptionBlockLight(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, conv_type="separable"):
        super().__init__()
        ch_1x1    = out_ch // 4
        remaining = out_ch - ch_1x1
        ch_3x3    = remaining // 3
        ch_5x5    = remaining // 3
        ch_pool   = out_ch - (ch_1x1 + ch_3x3 + ch_5x5)
        assert ch_pool > 0, f"channel split invalid: ch_pool={ch_pool} for out_ch={out_ch}"

        self.path1 = nn.Sequential(
            nn.Conv2d(in_ch, ch_1x1, 1, stride, 0, bias=False),
            nn.BatchNorm2d(ch_1x1),
            nn.ReLU(inplace=True),
        )

        # path2/path3 are where conv_type actually matters -- the 3x3 and
        # 5x5 "receptive field" branches of the Inception block. path1
        # (1x1) and path4 (pool projection) stay plain 1x1 convs regardless
        # of conv_type, same as in the original ablation.
        self.path2 = nn.Sequential(
            make_conv(in_ch, ch_3x3, 3, stride, 1, conv_type=conv_type),
            nn.BatchNorm2d(ch_3x3), nn.ReLU(inplace=True),
        )
        self.path3 = nn.Sequential(
            make_conv(in_ch, ch_5x5, 5, stride, 2, conv_type=conv_type),
            nn.BatchNorm2d(ch_5x5), nn.ReLU(inplace=True),
        )

        self.path4 = nn.Sequential(
            nn.MaxPool2d(3, stride, 1),
            nn.Conv2d(in_ch, ch_pool, 1, bias=False),
            nn.BatchNorm2d(ch_pool), nn.ReLU(inplace=True),
        )

        self.project = nn.Conv2d(out_ch, out_ch, 1, bias=False)

    def forward(self, x):
        out = torch.cat(
            [self.path1(x), self.path2(x), self.path3(x), self.path4(x)], dim=1
        )
        return self.project(out)


class PlainConvBlock(nn.Module):
    """Single-branch replacement for InceptionBlockLight, used for the
    'no inception' ablation. conv_type stays orthogonal so the same flag
    still applies."""

    def __init__(self, in_ch, out_ch, stride=1, conv_type="separable"):
        super().__init__()
        self.conv = make_conv(in_ch, out_ch, 3, stride, 1, conv_type=conv_type)

    def forward(self, x):
        return self.conv(x)


In [ ]:
class ResNetBlock(nn.Module):
    """Residual block wrapping two feature blocks. use_skip is the ablation
    toggle for the residual connection itself. conv_type selects which
    convolution operator (see the literature note above) is used inside
    the wrapped Inception / plain-conv block."""

    def __init__(
        self,
        in_ch,
        out_ch,
        stride=1,
        drop_path_prob=0.0,
        use_inception=True,
        conv_type="separable",
        use_skip=True,
    ):
        super().__init__()
        self.use_skip = use_skip
        block_kwargs = dict(conv_type=conv_type)
        block_cls = InceptionBlockLight if use_inception else PlainConvBlock

        self.feat1 = block_cls(in_ch, out_ch, stride, **block_kwargs)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.feat2 = block_cls(out_ch, out_ch, 1, **block_kwargs)
        self.bn2 = nn.BatchNorm2d(out_ch)

        if use_skip:
            if in_ch != out_ch or stride != 1:
                self.adjust = nn.Sequential(
                    nn.Conv2d(in_ch, out_ch, 1, stride, bias=False),
                    nn.BatchNorm2d(out_ch),
                )
            else:
                self.adjust = nn.Identity()
        else:
            self.adjust = None

        self.drop_path = DropPath(drop_path_prob)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.feat1(x)
        out = self.act(self.bn1(out))
        out = self.feat2(out)
        out = self.bn2(out)
        out = self.drop_path(out)
        if self.use_skip:
            out = out + self.adjust(x)
        return self.act(out)


## 4. Full model

Same architecture as the base model in the main notebook (with Squeeze-and-Excitation removed here), but every remaining component under study is switchable via constructor flags, including `conv_type` for the convolution-variant sweep. `num_classes` defaults to 100 here for CIFAR-100.

In [ ]:
class AblationResNet(BaseModel):
    """Same architecture as the original "MBInception" model from the main
    notebook (Squeeze-and-Excitation removed), with every remaining
    component under study switchable via constructor flags."""

    def __init__(
        self,
        in_ch=3,
        num_classes=100,
        base_filters=64,
        num_blocks=(2, 2, 2),
        drop_path_rate=0.1,
        use_inception=True,
        conv_type="separable",
        use_skip=True,
    ):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, base_filters, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(base_filters),
            nn.ReLU(inplace=True),
        )

        stages = []
        channels = base_filters
        total_blocks = sum(num_blocks)
        block_idx = 0
        for stage_idx, n_blocks in enumerate(num_blocks):
            out_ch = base_filters * (2 ** stage_idx)
            for b in range(n_blocks):
                stride = 2 if (stage_idx > 0 and b == 0) else 1
                this_drop = drop_path_rate * (block_idx / max(1, total_blocks - 1))
                stages.append(
                    ResNetBlock(
                        channels,
                        out_ch,
                        stride,
                        this_drop,
                        use_inception=use_inception,
                        conv_type=conv_type,
                        use_skip=use_skip,
                    )
                )
                channels = out_ch
                block_idx += 1

        self.stages = nn.Sequential(*stages)
        self.conv3x3 = nn.Sequential(
            nn.Conv2d(channels, channels, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(channels, num_classes)
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        out = self.stem(x)
        out = self.stages(out)
        out = self.conv3x3(out)
        out = self.pool(out).view(out.size(0), -1)
        out = self.dropout(out)
        out = self.fc(out)
        return out


## 5. Ablation configurations

11 ablation configs. The original 4-way structural study (Inception / skip / drop-path) is kept, but the single `no_separable_conv` toggle is replaced with a full sweep over `conv_type` (the literature variants defined in Section 3), so the separable convolution used in `full_model` is compared against six alternatives instead of just "off". Squeeze-and-Excitation has been removed from the architecture, so it's no longer one of the study variables.

In [ ]:
FULL = dict(use_inception=True, conv_type="separable", use_skip=True, drop_path_rate=0.1)

ABLATIONS = {
    # -- baseline --
    "full_model": dict(FULL),

    # -- convolution-type sweep (this is the literature comparison) --
    "conv_regular": {**FULL, "conv_type": "regular"},
    "conv_grouped_resnext": {**FULL, "conv_type": "grouped"},
    "conv_mixconv": {**FULL, "conv_type": "mixconv"},
    "conv_asymmetric_factorized": {**FULL, "conv_type": "asymmetric"},
    "conv_ghost": {**FULL, "conv_type": "ghost"},
    "conv_bsconv": {**FULL, "conv_type": "bsconv"},

    # -- original structural ablations (unchanged) --
    "no_skip_connection": {**FULL, "use_skip": False},
    "no_inception_(plain_conv)": {**FULL, "use_inception": False},
    "no_drop_path": {**FULL, "drop_path_rate": 0.0},
    "plain_cnn_baseline_(all_off)": {
        "use_inception": False,
        "conv_type": "regular",
        "use_skip": False,
        "drop_path_rate": 0.0,
    },
}

for name, flags in ABLATIONS.items():
    print(f"{name:32s} {flags}")


## 6. Data

CIFAR-100, with CIFAR-100's own normalization stats (different from CIFAR-10's). A 5,000-image split is held out from the training set for validation during training; the real test set is touched only once, at the end, per ablation.


In [ ]:
CIFAR100_MEAN = (0.5071, 0.4865, 0.4409)
CIFAR100_STD = (0.2673, 0.2564, 0.2762)


def get_dataloaders(data_dir, batch_size, val_size=5000, quick=False, num_workers=2, seed=42):
    train_transform = tt.Compose(
        [
            tt.RandomCrop(32, padding=4),
            tt.RandomHorizontalFlip(),
            tt.ToTensor(),
            tt.Normalize(CIFAR100_MEAN, CIFAR100_STD),
        ]
    )
    eval_transform = tt.Compose([tt.ToTensor(), tt.Normalize(CIFAR100_MEAN, CIFAR100_STD)])

    full_train = torchvision.datasets.CIFAR100(
        root=data_dir, train=True, download=True, transform=train_transform
    )
    # Separate copy with eval-time transform, so the held-out val split isn't augmented.
    full_train_eval = torchvision.datasets.CIFAR100(
        root=data_dir, train=True, download=True, transform=eval_transform
    )
    test_set = torchvision.datasets.CIFAR100(
        root=data_dir, train=False, download=True, transform=eval_transform
    )

    n_total = len(full_train)
    gen = torch.Generator().manual_seed(seed)
    perm = torch.randperm(n_total, generator=gen).tolist()
    val_idx, train_idx = perm[:val_size], perm[val_size:]

    train_set = Subset(full_train, train_idx)
    val_set = Subset(full_train_eval, val_idx)

    if quick:
        train_set = Subset(train_set, list(range(min(2000, len(train_set)))))
        val_set = Subset(val_set, list(range(min(500, len(val_set)))))
        test_set = Subset(test_set, list(range(min(500, len(test_set)))))

    train_dl = DataLoader(
        train_set, batch_size=batch_size, shuffle=True, num_workers=num_workers,
        pin_memory=True, drop_last=True,
    )
    val_dl = DataLoader(
        val_set, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True
    )
    test_dl = DataLoader(
        test_set, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True
    )
    return train_dl, val_dl, test_dl

## 7. Training / evaluation utilities

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def accuracy(predicted, actual):
    _, preds = torch.max(predicted, dim=1)
    return torch.tensor(torch.sum(preds == actual).item() / len(preds))


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    outputs = []
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        outputs.append(model.validation_step((images, targets)))
    return model.validation_epoch_end(outputs)


def fit(model, train_dl, val_dl, optimizer, epochs, device, grad_clip=0.1, scheduler=None):
    """OneCycleLR stepped per batch, optional gradient clipping, per-epoch
    train/val loss & accuracy logged."""
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}

    for epoch in range(epochs):
        model.train()
        train_losses, train_accs = [], []

        for images, targets in train_dl:
            images, targets = images.to(device), targets.to(device)

            outputs = model(images)
            loss = F.cross_entropy(outputs, targets)

            optimizer.zero_grad()
            loss.backward()
            if grad_clip:
                torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip)
            optimizer.step()

            if scheduler:
                scheduler.step()  # OneCycleLR requires a per-batch step

            train_losses.append(loss.detach())
            train_accs.append(accuracy(outputs, targets))

        val_metrics = evaluate(model, val_dl, device)

        train_loss_avg = torch.stack(train_losses).mean().item()
        train_acc_avg = torch.stack(train_accs).mean().item()
        lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_loss_avg)
        history["train_acc"].append(train_acc_avg)
        history["val_loss"].append(val_metrics["val_loss"])
        history["val_acc"].append(val_metrics["val_acc"])
        history["lr"].append(lr)

        print(
            f"epoch {epoch + 1}/{epochs} | "
            f"train_loss={train_loss_avg:.4f} train_acc={train_acc_avg:.4f} | "
            f"val_loss={val_metrics['val_loss']:.4f} val_acc={val_metrics['val_acc']:.4f} | "
            f"lr={lr:.6f}"
        )

    return history


def run_ablation(name, model_kwargs, loaders, device, epochs, max_lr, weight_decay, grad_clip, pct_start, seed=42):
    train_dl, val_dl, test_dl = loaders
    set_seed(seed)
    model = AblationResNet(**model_kwargs).to(device)
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    optimizer = torch.optim.Adam(model.parameters(), lr=max_lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lr,
        epochs=epochs,
        steps_per_epoch=len(train_dl),
        pct_start=pct_start,
        div_factor=1,
    )

    start = time.time()
    history = fit(model, train_dl, val_dl, optimizer, epochs, device, grad_clip=grad_clip, scheduler=scheduler)
    elapsed = time.time() - start

    test_metrics = evaluate(model, test_dl, device)

    return {
        "name": name,
        "num_params": num_params,
        "final_val_acc": history["val_acc"][-1],
        "best_val_acc": max(history["val_acc"]),
        "final_val_loss": history["val_loss"][-1],
        "test_acc": test_metrics["val_acc"],
        "test_loss": test_metrics["val_loss"],
        "train_time_sec": elapsed,
        "history": history,
    }

## 8. Reporting utilities

In [ ]:
def save_results(results, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    csv_path = os.path.join(output_dir, "results.csv")
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(
            ["name", "num_params", "final_val_acc", "best_val_acc",
             "final_val_loss", "test_acc", "test_loss", "train_time_sec"]
        )
        for r in results:
            writer.writerow(
                [r["name"], r["num_params"], r["final_val_acc"], r["best_val_acc"],
                 r["final_val_loss"], r["test_acc"], r["test_loss"], r["train_time_sec"]]
            )

    json_path = os.path.join(output_dir, "histories.json")
    with open(json_path, "w") as f:
        json.dump({r["name"]: r["history"] for r in results}, f, indent=2)

    print(f"Saved: {csv_path}")
    print(f"Saved: {json_path}")


def plot_results(results, output_dir):
    plt.rcParams.update({
        "font.size": 10, "font.family": "serif",
        "axes.labelsize": 10, "axes.titlesize": 12,
        "legend.fontsize": 9, "xtick.labelsize": 9, "ytick.labelsize": 9,
    })

    # 1. val accuracy curves
    plt.figure(figsize=(9, 6))
    for r in results:
        plt.plot(r["history"]["val_acc"], label=r["name"])
    plt.xlabel("Epoch", fontweight="bold")
    plt.ylabel("Validation Accuracy", fontweight="bold")
    plt.title("Validation Accuracy per Ablation (CIFAR-100)")
    plt.legend(fontsize=8)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "val_acc_curves.png"), dpi=150)
    plt.show()

    # 2. bar chart of best val accuracy
    ordered = sorted(results, key=lambda r: r["best_val_acc"], reverse=True)
    names = [r["name"] for r in ordered]
    accs = [r["best_val_acc"] for r in ordered]

    plt.figure(figsize=(9, 6))
    plt.bar(names, accs, color="steelblue")
    plt.ylabel("Best Validation Accuracy", fontweight="bold")
    plt.title("Best Validation Accuracy by Ablation (CIFAR-100)")
    plt.xticks(rotation=30, ha="right", fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "final_accuracy_bar.png"), dpi=150)
    plt.show()

    # 3. params vs accuracy scatter
    plt.figure(figsize=(8, 6))
    for r in results:
        plt.scatter(r["num_params"], r["best_val_acc"])
        plt.annotate(r["name"], (r["num_params"], r["best_val_acc"]), fontsize=7)
    plt.xlabel("Parameter Count", fontweight="bold")
    plt.ylabel("Best Validation Accuracy", fontweight="bold")
    plt.title("Parameter Efficiency (CIFAR-100)")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "params_vs_accuracy.png"), dpi=150)
    plt.show()

    print(f"Saved plots to: {output_dir}")


def print_summary_table(results):
    ordered = sorted(results, key=lambda r: r["best_val_acc"], reverse=True)
    print("=" * 105)
    print(
        f"{'Ablation':32s} {'Params':>10s} {'Best Val Acc':>13s} "
        f"{'Final Val Acc':>14s} {'Test Acc':>10s} {'Time (s)':>10s}"
    )
    print("-" * 105)
    for r in ordered:
        print(
            f"{r['name']:32s} {r['num_params']:>10,d} {r['best_val_acc']:>13.4f} "
            f"{r['final_val_acc']:>14.4f} {r['test_acc']:>10.4f} {r['train_time_sec']:>10.1f}"
        )
    print("=" * 105)

## 9. Quick test (run this first)

Tiny data subset, 2 epochs, every config in `ABLATIONS` -- just to confirm each ablation (including all 7 `conv_type` variants) trains end-to-end without errors before you commit to the full per-config runs below. This whole cell is cheap; it does **not** save any results.

In [ ]:
# Quick smoke test: tiny data subset, 2 epochs, every config in ABLATIONS.
# Does NOT save anything to disk -- it only confirms every config builds
# and trains end-to-end without crashing before you commit to the real
# per-config runs in Section 11. Uses a small model (fewer filters/blocks)
# purely for speed; this is not meant to produce meaningful accuracy numbers.

_quick_train_dl, _quick_val_dl, _quick_test_dl = get_dataloaders(
    "./data", batch_size=32, quick=True, num_workers=0, seed=42
)
_quick_base_arch = dict(in_ch=3, num_classes=100, base_filters=16, num_blocks=(1, 1, 1))

print(f"Running quick smoke test on {len(ABLATIONS)} configs (2 epochs each, tiny data)...\n")

_quick_failed = []
for _name, _flags in ABLATIONS.items():
    print(f"--- {_name} ---")
    try:
        set_seed(0)
        _model_kwargs = {**_quick_base_arch, **_flags}
        _model = AblationResNet(**_model_kwargs).to(device)
        _optimizer = torch.optim.Adam(_model.parameters(), lr=1e-3)
        _scheduler = torch.optim.lr_scheduler.OneCycleLR(
            _optimizer,
            max_lr=1e-3,
            epochs=2,
            steps_per_epoch=len(_quick_train_dl),
            pct_start=0.45,
            div_factor=1,
        )
        fit(
            _model, _quick_train_dl, _quick_val_dl, _optimizer,
            epochs=2, device=device, grad_clip=0.1, scheduler=_scheduler,
        )
        evaluate(_model, _quick_test_dl, device)
        print(f"[OK] {_name}\n")
    except Exception as e:
        _quick_failed.append(_name)
        print(f"[FAILED] {_name}: {e}\n")

if _quick_failed:
    raise RuntimeError(f"Quick test failed for: {_quick_failed}")
print(f"All {len(ABLATIONS)} configs passed the quick test -- safe to move on to Section 10/11.")


## 10. Shared setup for the real per-config runs

Same hyperparameters as before (batch 64, Adam, OneCycleLR `max_lr=0.001` / `pct_start=0.45`, grad clip 0.1). Run this cell once per session — it builds the real dataloaders and defines `run_and_save_one`, which trains a **single** ablation config and writes its result straight to disk under `PER_CONFIG_DIR`. If a result file for a config already exists, it's loaded from disk instead of retrained (set `force=True` to override), so re-running a cell, or resuming after a crash/restart, is safe and won't waste time redoing finished configs.

In [ ]:
EPOCHS = 30
BATCH_SIZE = 64
MAX_LR = 0.001
PCT_START = 0.45
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 0.1
BASE_FILTERS = 64
NUM_BLOCKS = (2, 2, 2)
NUM_CLASSES = 100
SEEDS = [42]  # add more seeds, e.g. [42, 123, 7], to average out run-to-run noise
OUTPUT_DIR = "./ablation_results_cifar100"
PER_CONFIG_DIR = os.path.join(OUTPUT_DIR, "per_config")
os.makedirs(PER_CONFIG_DIR, exist_ok=True)

loaders = get_dataloaders("./data", batch_size=BATCH_SIZE, seed=42)
base_arch = dict(in_ch=3, num_classes=NUM_CLASSES, base_filters=BASE_FILTERS, num_blocks=NUM_BLOCKS)


def _result_path(name):
    safe = name.replace("/", "_")
    return os.path.join(PER_CONFIG_DIR, f"{safe}.json")


def run_and_save_one(name, force=False):
    """Train ONE ablation config and save its result to its own file.

    Safe to call repeatedly: if `name`'s result already exists on disk this just
    loads and returns it, unless force=True.
    """
    if name not in ABLATIONS:
        raise KeyError(f"Unknown ablation '{name}'. Options: {list(ABLATIONS.keys())}")

    path = _result_path(name)
    if os.path.exists(path) and not force:
        print(f"{name}: already done, loading saved result from {path} (pass force=True to rerun)")
        with open(path) as f:
            return json.load(f)

    flags = ABLATIONS[name]
    model_kwargs = {**base_arch, **flags}

    seed_runs = []
    for seed in SEEDS:
        res = run_ablation(
            name, model_kwargs, loaders, device,
            EPOCHS, MAX_LR, WEIGHT_DECAY, GRAD_CLIP, PCT_START, seed=seed,
        )
        seed_runs.append(res)

    if len(seed_runs) == 1:
        result = seed_runs[0]
    else:
        result = dict(seed_runs[0])
        for key in ["final_val_acc", "best_val_acc", "final_val_loss", "test_acc", "test_loss", "train_time_sec"]:
            result[key] = sum(r[key] for r in seed_runs) / len(seed_runs)

    with open(path, "w") as f:
        json.dump(result, f, indent=2)
    print(f"{name}: done, saved to {path}")
    return result

## 11. Run each ablation (one cell each -- run these separately)

Run Sections 1-10 once first (they define the model, data, and the `run_and_save_one` helper). Then run the 11 cells below **one at a time**, in whatever order and however many sessions you like. Each cell trains a single config and immediately writes its result to `ablation_results_cifar100/per_config/`, so:\n\n- if your computer can only handle one config per session, run one cell, close everything, and come back later for the next\n- if a run crashes partway, the configs that already finished are untouched -- just rerun Sections 1-10 and the remaining config cells\n- re-running a cell that already finished is a no-op (it just reloads the saved result) unless you pass `force=True`\n\nThe first seven configs (`full_model` through `conv_bsconv`) all use the same Inception + skip + drop-path settings and differ only in `conv_type` -- that block is the actual literature comparison. The remaining four repeat the original structural ablations.

### Ablation 1 of 11 -- `full_model`

In [ ]:
result_full_model = run_and_save_one("full_model")


### Ablation 2 of 11 -- `conv_regular`

In [ ]:
result_conv_regular = run_and_save_one("conv_regular")


### Ablation 3 of 11 -- `conv_grouped_resnext`

In [ ]:
result_conv_grouped_resnext = run_and_save_one("conv_grouped_resnext")


### Ablation 4 of 11 -- `conv_mixconv`

In [ ]:
result_conv_mixconv = run_and_save_one("conv_mixconv")


### Ablation 5 of 11 -- `conv_asymmetric_factorized`

In [ ]:
result_conv_asymmetric_factorized = run_and_save_one("conv_asymmetric_factorized")


### Ablation 6 of 11 -- `conv_ghost`

In [ ]:
result_conv_ghost = run_and_save_one("conv_ghost")


### Ablation 7 of 11 -- `conv_bsconv`

In [ ]:
result_conv_bsconv = run_and_save_one("conv_bsconv")


### Ablation 8 of 11 -- `no_skip_connection`

In [ ]:
result_no_skip_connection = run_and_save_one("no_skip_connection")


### Ablation 9 of 11 -- `no_inception_(plain_conv)`

In [ ]:
result_no_inception_plain_conv = run_and_save_one("no_inception_(plain_conv)")


### Ablation 10 of 11 -- `no_drop_path`

In [ ]:
result_no_drop_path = run_and_save_one("no_drop_path")


### Ablation 11 of 11 -- `plain_cnn_baseline_(all_off)`

In [ ]:
result_plain_cnn_baseline_all_off = run_and_save_one("plain_cnn_baseline_(all_off)")


## 12. Aggregate results & plots (run after all configs are done)

Loads whatever's been saved to `PER_CONFIG_DIR` so far. If all configs are present it builds the same comparison CSV/plots/summary table as before; if some are still missing it tells you which ones and stops (run those cells in Section 11, then re-run this cell).


In [ ]:
all_names = list(ABLATIONS.keys())
results = []
missing = []
for name in all_names:
    path = _result_path(name)
    if os.path.exists(path):
        with open(path) as f:
            results.append(json.load(f))
    else:
        missing.append(name)

if missing:
    print(f"Missing {len(missing)}/{len(all_names)} configs, run their cells in Section 11 first:")
    for name in missing:
        print(f"  - {name}")

if results:
    save_results(results, OUTPUT_DIR)
if not missing:
    plot_results(results, OUTPUT_DIR)
    print_summary_table(results)
elif results:
    print(f"\n({len(results)}/{len(all_names)} configs available — plots/summary skipped until all are done. "
          f"results.csv/histories.json were still updated with what's available.)")